In [1]:
import torch, platform, sys
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


In [2]:
!pip -q install "transformers==4.44.2" "datasets>=2.20" "accelerate>=0.34" "evaluate>=0.4" "tokenizers>=0.19"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.1 MB/s eta 0:00:00


In [4]:
TRAIN_CSV = "/content/reddit_bert_train.csv"
VAL_CSV   = "/content/reddit_bert_val.csv"
TEST_CSV  = "/content/reddit_bert_test.csv"

OUTPUT_DIR = "/content/outputs/bert-base-uncased"


In [5]:
import pandas as pd

for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    df = pd.read_csv(path)
    print(path, df.shape)
    print(df.head(2))


/content/reddit_bert_train.csv (87106, 3)
                                        cleaned_text subreddit  label
0  Audio books while ADHD I listen to a lot of au...      adhd      1
1  Failed trial of Concerta the crushing disappoi...      adhd      1
/content/reddit_bert_val.csv (18666, 3)
                                        cleaned_text subreddit  label
0  Would like to share a few small things that he...      adhd      1
1  ADD woes My mom Im upstairs Can you throw down...      adhd      1
/content/reddit_bert_test.csv (18668, 3)
                                        cleaned_text subreddit  label
0  How to stop losing things Hey my lovely ADHDer...      adhd      1
1  I dont know how to function without meds Docto...      adhd      1


In [6]:
import os, json, random, numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)
import evaluate

# -------------------
# Config
# -------------------
MODEL_NAME = "bert-base-cased"
TEXT_COL   = "cleaned_text"
LABEL_COL  = "label"
SEED       = 42
random.seed(SEED); np.random.seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------
# Load data
# -------------------
dataset = load_dataset("csv",
    data_files={"train": TRAIN_CSV, "validation": VAL_CSV, "test": TEST_CSV}
)

# Ensure numeric labels
def coerce_label(example):
    example[LABEL_COL] = int(example[LABEL_COL])
    return example
dataset = dataset.map(coerce_label)

# Determine number of labels
unique_labels = sorted(set(dataset["train"][LABEL_COL]))
num_labels = len(unique_labels)
print("num_labels:", num_labels, "labels:", unique_labels)

# -------------------
# Tokenizer / preprocess
# -------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess(batch):
    return tok(batch[TEXT_COL], truncation=True, max_length=256)
tokenized = dataset.map(preprocess, batched=True, remove_columns=[TEXT_COL])

# -------------------
# Metrics
# -------------------
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
        "precision":precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall":   recall.compute(predictions=preds, references=labels, average="weighted")["recall"],
    }

# -------------------
# Model
# -------------------
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# Mixed precision if supported (bf16 on A100, else fp16 on T4/V100)
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
fp16_ok = torch.cuda.is_available()

# -------------------
# Trainer args
# -------------------
args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    seed=SEED,
    report_to=["none"],
    fp16=(fp16_ok and not bf16_ok),
    bf16=bf16_ok,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tok,
    data_collator=DataCollatorWithPadding(tokenizer=tok),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# Save the final/best model + tokenizer (outside checkpoints)
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

# Persist training args and label mapping
with open(f"{OUTPUT_DIR}/training_args.json", "w") as f:
    json.dump(vars(args), f, default=str, indent=2)
with open(f"{OUTPUT_DIR}/label_space.json", "w") as f:
    json.dump({"labels": list(map(int, unique_labels))}, f, indent=2)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/87106 [00:00<?, ? examples/s]

Map:   0%|          | 0/18666 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/18666 [00:00<?, ? examples/s]

Map:   0%|          | 0/18668 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.126700,0.135769,0.966088,0.966161,0.966474,0.966088
2,0.081500,0.126790,0.969570,0.969494,0.969680,0.969570
3,0.004800,0.171190,0.969999,0.970043,0.970197,0.969999
4,0.009900,0.199423,0.970213,0.970247,0.970346,0.970213
5,0.002900,0.233607,0.971285,0.971295,0.971311,0.971285


import pandas as pd

for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    df = pd.read_csv(path)
    print(path, df.shape)
    print(df.head(2))


In [7]:
# ----- paths (EDIT THESE) -----
OUTPUT_DIR = "/content/outputs/bert-base-uncased"  # where the trained Reddit model is saved
TWITTER_CSV = "/content/twitter_bert.csv"

# ----- columns from file -----
TEXT_COL  = "cleaned_text"
LABEL_COL = "label"
MAX_LEN   = 128


In [9]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "evaluate>=0.4" accelerate scikit-learn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding
from datasets import load_dataset
import numpy as np, pandas as pd, os, json
from scipy.special import softmax
import evaluate

# Load model + tokenizer trained on Reddit
tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)

# Load Twitter CSV
twitter_ds = load_dataset("csv", data_files={"twitter": TWITTER_CSV})["twitter"]

# Ensure numeric labels
if LABEL_COL and LABEL_COL in twitter_ds.column_names:
    twitter_ds = twitter_ds.map(lambda ex: {**ex, LABEL_COL: int(ex[LABEL_COL])})

# Tokenize (keep same max_length as training)
def preprocess(b):
    return tok(b[TEXT_COL], truncation=True, max_length=MAX_LEN)
tok_tw = twitter_ds.map(preprocess, batched=True)

# Predict
trainer = Trainer(model=model, tokenizer=tok, data_collator=DataCollatorWithPadding(tokenizer=tok))
pred_out = trainer.predict(tok_tw.remove_columns([c for c in [TEXT_COL] if c in tok_tw.column_names]))
logits = pred_out.predictions
y_pred = logits.argmax(axis=-1)
probs  = softmax(logits, axis=-1)

# Save predictions
TW_OUT = os.path.join(OUTPUT_DIR, "twitter_eval")
os.makedirs(TW_OUT, exist_ok=True)

df = pd.DataFrame({"text": twitter_ds[TEXT_COL], "pred_label": y_pred})
for i in range(probs.shape[1]):
    df[f"prob_{i}"] = probs[:, i]
df.to_csv(os.path.join(TW_OUT, "twitter_predictions.csv"), index=False)

# Metrics
metrics = {}
if LABEL_COL and LABEL_COL in twitter_ds.column_names:
    y_true = np.array(twitter_ds[LABEL_COL])
    acc = evaluate.load("accuracy").compute(predictions=y_pred, references=y_true)["accuracy"]
    f1  = evaluate.load("f1").compute(predictions=y_pred, references=y_true, average="weighted")["f1"]
    pr  = evaluate.load("precision").compute(predictions=y_pred, references=y_true, average="weighted")["precision"]
    rc  = evaluate.load("recall").compute(predictions=y_pred, references=y_true, average="weighted")["recall"]
    metrics.update(dict(accuracy=float(acc), f1=float(f1), precision=float(pr), recall=float(rc)))

    #  ROC-AUC for binary
    from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, probs[:, 1]))
    except Exception:
        pass

    with open(os.path.join(TW_OUT, "metrics_twitter.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    # Save detailed reports
    rep = classification_report(y_true, y_pred, digits=4)
    with open(os.path.join(TW_OUT, "classification_report_twitter.txt"), "w") as f:
        f.write(rep)
    cm = confusion_matrix(y_true, y_pred)
    np.savetxt(os.path.join(TW_OUT, "confusion_matrix_twitter.csv"), cm, fmt="%d", delimiter=",")

print("Saved:", os.path.join(TW_OUT, "twitter_predictions.csv"))
if metrics:
    print("Twitter metrics:", metrics)


Generating twitter split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: arefeh-emrani (arefeh-emrani-concordia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Saved: /content/outputs/bert-base-uncased/twitter_eval/twitter_predictions.csv
Twitter metrics: {'accuracy': 0.56685, 'f1': 0.5191531131246087, 'precision': 0.6108208218946269, 'recall': 0.56685, 'roc_auc': 0.590440325}
